# Multi-Reference Facial Inpainting: Research-Validation Phase
## Step 150 (Baseline) vs Step 500 (Cloud-Trained)

This notebook executes the full research-validation phase end-to-end on Kaggle NVIDIA T4 GPU:
1. **Reranker Leakage Audit**: Formally audits Ablation A7 for ground-truth target leakage.
2. **Side-by-Side Visual Grid**: Generates 10-panel comparative grid, zoomed crops, and difference heatmaps.
3. **Multi-Severity Suite**: Evaluates 25%, 50%, 75%, and 100% full-face occlusion tiers.
4. **Multi-Subject Statistical Benchmark**: Evaluates 10 diverse subjects with paired t-tests and statistical aggregation.
5. **Artifact Packaging**: Bundles all generated images, CSVs, and JSON reports for 1-click download.

In [ ]:
# [1/8] GPU & CUDA Verification
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU:      {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:      {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    raise RuntimeError("Kaggle GPU Accelerator is required! Enable GPU T4 in Settings panel on the right.")

In [ ]:
# [2/9] Install Dependencies & Unpack Research Validation Package
!pip install -q diffusers transformers accelerate mediapipe opencv-python onnxruntime-gpu scipy scikit-image facenet-pytorch piq

# Unpack package (if uploaded as archive)
!if [ -f "research_validation_package.zip" ]; then unzip -o research_validation_package.zip; fi
!if [ -f "dist/research_validation_package.zip" ]; then unzip -o dist/research_validation_package.zip; fi


In [ ]:
# [3/8] Audit Reranker / Ablation A7 for Evaluation Leakage
!python scripts/audit_reranker_leakage.py

In [ ]:
# [4/8] Generate Side-by-Side Visual Outputs & Difference Heatmap (50 DDIM Steps)
!python scripts/generate_visual_comparison.py \
    --step150 checkpoints/baselines/step_150_best/step_150.pt \
    --step500 checkpoints/cloud_training/step_500.pt \
    --device cuda \
    --steps 50 \
    --output_dir evaluation/visualizations

In [ ]:
# [5/8] Run Multi-Severity Benchmark Suite (25%, 50%, 75%, 100% Full-Face)
!python scripts/evaluate_severity_suite.py \
    --checkpoints checkpoints/baselines/step_150_best/step_150.pt checkpoints/cloud_training/step_500.pt \
    --device cuda \
    --steps 50 \
    --output_dir evaluation/reports

In [ ]:
# [6/8] Run Multi-Subject Statistical Benchmark (10 Subjects)
!python scripts/evaluate_multi_subject.py \
    --checkpoints checkpoints/baselines/step_150_best/step_150.pt checkpoints/cloud_training/step_500.pt \
    --device cuda \
    --steps 50 \
    --output_dir evaluation/reports

In [ ]:
# [7/9] Run Clean O3 Verifier-Aware Reranker Benchmark (Zero Leakage, K=3)
!python scripts/audit_clean_reranker.py
!python scripts/evaluate_clean_o3_reranker.py \
    --checkpoints checkpoints/baselines/step_150_best/step_150.pt checkpoints/cloud_training/step_500.pt \
    --device cuda \
    --steps 50 \
    --seeds 42 43 44


In [ ]:
# [7/8] Display 10-Panel Comparative Grid Inline
from IPython.display import Image, display
grid_path = "evaluation/visualizations/03_side_by_side_comparison.png"
if os.path.exists(grid_path):
    display(Image(filename=grid_path, width=1200))
else:
    print("Grid not generated yet.")

In [ ]:
# [8/8] Package All Benchmark Results & Reports for 1-Click Download
!zip -r research_validation_results.zip evaluation/reports/ evaluation/visualizations/
print("All results packed into research_validation_results.zip ready for download!")